## Prepare dataset in the required format

### image url

The met image urls are not publicly accessible. The images paths on google colab are not https public urls. So upload the images to github and then re-extract the public https urls.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

src_folder = "/content/drive/MyDrive/master_thesis_project/met_images"

# Get all .jpg files
files = [f for f in os.listdir(src_folder) if f.lower().endswith(".jpg")]

len(files)

9872

In [ ]:
!pip install --upgrade transformers huggingface_hub

In [ ]:
import huggingface_hub
print(huggingface_hub.__version__)

0.36.0


In [ ]:
from huggingface_hub import HfApi, login

# Log in (paste HF token once)
login()

api = HfApi()

In [ ]:
# check whether all the images files are uploaded
def verify_actual_file_count(repo_id):
    """Verify the actual file count and check if there are subfolders"""

    print(f"Verifying actual file count in: {repo_id}")

    try:
        # Get all files in the repo
        repo_files = list(api.list_repo_files(repo_id=repo_id, repo_type="dataset"))
        print(f"Total files reported by API: {len(repo_files)}")

        # Check if files are organized in subfolders
        from collections import defaultdict
        folder_structure = defaultdict(list)

        for file_path in repo_files:
            if '/' in file_path:
                folder = file_path.split('/')[0]
                folder_structure[folder].append(file_path)
            else:
                folder_structure['root'].append(file_path)

        print(f"Folder structure:")
        for folder, files in folder_structure.items():
            print(f"  {folder}: {len(files)} files")

        # Show some file examples from different folders
        print(f"\nSample files:")
        for folder, files in list(folder_structure.items())[:3]:  # Show first 3 folders
            print(f"  {folder}/: {files[:3]}")  # Show first 3 files in each folder

        return len(repo_files)

    except Exception as e:
        print(f"Error: {e}")
        return 0

# Check what's actually there
repo_id = "Huan3/met_artefacts_images"
actual_count = verify_actual_file_count(repo_id)

Verifying actual file count in: Huan3/met_artefacts_images
Total files reported by API: 9873
Folder structure:
  root: 9873 files

Sample files:
  root/: ['.gitattributes', '239585_primary.jpg', '239586_primary.jpg']


Git counts hidden files too.  
When initializing the repo, Git created two hidden items automatically:  
.gitignore (maybe, depending on settings).
.gitattributes or .git folder internals.
Those can account for the 2 extra objects — so 4675 images + 2 metadata files = 4677 total objects.

generate https publicly accessible image paths

In [ ]:
import os

repo_base_url = "https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main"
repo_folder = "/content/drive/MyDrive/master_thesis_project/met_images"

image_urls = []

# Sort files for consistent ordering
for f in sorted(os.listdir(repo_folder)):
    if f.lower().endswith(".jpg"):
        image_urls.append(f"{repo_base_url}/{f}")

print(f"Generated {len(image_urls)} URLs")
print(f"Sample URL: {image_urls[0] if image_urls else 'No URLs'}")

Generated 9872 URLs
Sample URL: https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/239585_primary.jpg


In [ ]:
import random

random.sample(image_urls, 5) # show random 5 URLs

['https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/753033_additional_0.jpg',
 'https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/250966_primary.jpg',
 'https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/248663_primary.jpg',
 'https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/242800_additional_3.jpg',
 'https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/248588_primary.jpg']

https://techcommunity.microsoft.com/blog/azure-ai-foundry-blog/announcing-new-fine-tuning-capabilities-with-images-on-azure-openai-service/4303695?utm_source=chatgpt.com

Image Dataset Requirements.

To ensure the best performance and compliance, there are specific requirements for your image datasets:  

Size: Your training file can contain up to 50,000 examples with images, with each example having a maximum of 64 images. Each image can be up to 10 MB.  
Format: Images must be in JPEG, PNG, or WEBP format and in RGB or RGBA mode. Images cannot be included as output from messages with the assistant role.  
Content Moderation: Images are scanned before training to ensure compliance with our usage policy. Images containing people, faces, or CAPTCHAs will be excluded from the dataset.  

Handling Skipped Images  

If your images are skipped during the training process, it could be due to several reasons such as containing CAPTCHAs, people, faces, inaccessible URLs, large file sizes, invalid mode or invalid formats. Ensure your images meet the specified requirements to avoid these issues.  

Uploading Large Files  

For large training files, you can upload files up to 8 GB in multiple parts using the Uploads API. This is particularly useful for extensive datasets that exceed the 512 MB limit of the Files API.  

Reducing Training Costs  

To optimize training costs, you can set the detail parameter for an image to low, which resizes the image to 512 by 512 pixels and represents it by 85 tokens regardless of its size. This reduces the cost of training while maintaining the quality of the model.  

Additional Considerations  

To control the fidelity of image understanding, you can set the detail parameter of image_url to low, high, or auto for each image. This affects the number of tokens per image that the model sees during training and impacts the cost of training.  

We are thrilled to see how you will leverage these new capabilities to create innovative and engaging AI applications. For more detailed information, please refer to our documentation on Azure OpenAI Service.  

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from huggingface_hub import HfApi
from collections import defaultdict

repo_id = "Huan3/met_artefacts_images"

# Get all files from Hugging Face repository
repo_files = list(api.list_repo_files(repo_id=repo_id, repo_type="dataset"))

print(f"Total files in Hugging Face repository: {len(repo_files)}\n")

# Filter image files
image_files = [f for f in repo_files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]
print(f"Total supported image files: {len(image_files)}\n")

# Check for unsupported formats
for f in repo_files:
    if not f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
        print(f"Unsupported format: {f}")

# Check for very large number of images per artifact
artifact_counts = defaultdict(int)
for f in image_files:
    # Extract artifact ID from filename (format: id_primary.jpg or id_additional_0.jpg)
    artifact_id = f.split("_")[0]
    artifact_counts[artifact_id] += 1

print(f"\nArtifacts with more than 10 images:")
artifacts_over_limit = []
for artifact_id, count in artifact_counts.items():
    if count > 10:
        artifacts_over_limit.append((artifact_id, count))
        print(f"Artifact {artifact_id} has {count} images (consider reducing to 10)")

print(f"\n📊 Summary:")
print(f"Total artifacts: {len(artifact_counts)}")
print(f"Artifacts with >10 images: {len(artifacts_over_limit)}")
print(f"Average images per artifact: {len(image_files) / len(artifact_counts):.2f}")

Total files in Hugging Face repository: 9873

Total supported image files: 9872

Unsupported format: .gitattributes

Artifacts with more than 10 images:
Artifact 247017 has 15 images (consider reducing to 10)
Artifact 248129 has 12 images (consider reducing to 10)
Artifact 253348 has 14 images (consider reducing to 10)
Artifact 253370 has 16 images (consider reducing to 10)
Artifact 254843 has 14 images (consider reducing to 10)

📊 Summary:
Total artifacts: 5965
Artifacts with >10 images: 5
Average images per artifact: 1.65


In [ ]:
import requests
from PIL import Image
from io import BytesIO
from tqdm import tqdm
from huggingface_hub import HfApi

repo_id = "Huan3/met_artefacts_images"
api = HfApi()

print("Getting file list from Hugging Face...")
repo_files = list(api.list_repo_files(repo_id=repo_id, repo_type="dataset"))
image_files = [f for f in repo_files if f.lower().endswith('.jpg')]

print(f"Found {len(image_files)} images in repository")

too_large = []
base_url = f"https://huggingface.co/datasets/{repo_id}/resolve/main/"

print("Checking dimensions for ALL files...")
for file_name in tqdm(image_files, desc="Checking images", unit="file"):
    try:
        # Download and check image
        url = base_url + file_name
        response = requests.get(url, timeout=30)
        response.raise_for_status()

        # Open image from memory
        img = Image.open(BytesIO(response.content))
        width, height = img.size
        if width > 512 or height > 512:
            too_large.append((file_name, width, height))

    except Exception as e:
        print(f"Could not check {file_name}: {e}")

# Results
print(f"\nFINAL RESULTS:")
print(f"{len(too_large)} out of {len(image_files)} images are larger than 512px")
print(f"That's {len(too_large)/len(image_files)*100:.1f}% of all images")

if too_large:
    print(f"\nSample of large images:")
    for name, w, h in too_large[:10]:  # Show first 10
        print(f" - {name}: {w}x{h}px")

    # Show the largest ones
    too_large.sort(key=lambda x: max(x[1], x[2]), reverse=True)
    print(f"\nTop 5 largest images:")
    for name, w, h in too_large[:5]:
        max_dim = max(w, h)
        print(f" - {name}: {w}x{h}px (max: {max_dim}px)")

Getting file list from Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Found 9872 images in repository
Checking dimensions for ALL files...


Checking images: 100%|██████████| 9872/9872 [1:32:10<00:00,  1.79file/s]


FINAL RESULTS:
9823 out of 9872 images are larger than 512px
That's 99.5% of all images

Sample of large images:
 - 239585_primary.jpg: 3000x4000px
 - 239586_primary.jpg: 4000x4000px
 - 239588_primary.jpg: 4000x4000px
 - 239593_primary.jpg: 4000x4000px
 - 239596_primary.jpg: 4000x4000px
 - 239598_primary.jpg: 4000x4000px
 - 239706_primary.jpg: 4000x4000px
 - 239897_additional_0.jpg: 3000x4000px
 - 239897_additional_1.jpg: 800x1257px
 - 239897_primary.jpg: 3000x4000px

Top 5 largest images:
 - 241987_primary.jpg: 5667x3080px (max: 5667px)
 - 250999_additional_0.jpg: 5600x3360px (max: 5600px)
 - 250999_primary.jpg: 5560x3333px (max: 5560px)
 - 241919_primary.jpg: 5487x3527px (max: 5487px)
 - 239585_primary.jpg: 3000x4000px (max: 4000px)


when filtering in the csv file, the total numbers of artefatcs is 5727.   
The likely reasons for the discrepancy:    
CSV filtering: filters artifacts by what exists in the CSV     
Missing metadata: Some artifacts have images but no corresponding CSV entry    
Corrupted/invalid IDs: Some artifact IDs might not match between files and CSV    

In [ ]:
import json
import pandas as pd
import os

input_file = "/content/drive/My Drive/master_thesis_project/met_metadata_0509.csv"
output_file = "/content/drive/My Drive/master_thesis_project/met_data_for_gpt4o_description_only.jsonl"

# Hugging Face URL prefix
hf_prefix = "https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/"

# Read CSV file
df = pd.read_csv(input_file)

# Get all image files from Hugging Face to map artifact IDs to available images
repo_files = list(api.list_repo_files(repo_id="Huan3/met_artefacts_images", repo_type="dataset"))
image_files = [f for f in repo_files if f.lower().endswith('.jpg')]

# Create a mapping from artifact ID to its available images
artifact_images = {}
for image_file in image_files:
    artifact_id = image_file.split('_')[0]  # Extract ID from filename
    if artifact_id not in artifact_images:
        artifact_images[artifact_id] = []
    artifact_images[artifact_id].append(image_file)

# Sort images so primary comes first, then additional_0, additional_1, etc.
def sort_images(images):
    def sort_key(img):
        if '_primary' in img:
            return (0, 0)
        elif '_additional_' in img:
            try:
                num = int(img.split('_additional_')[1].split('.')[0])
                return (1, num)
            except:
                return (2, img)
        else:
            return (3, img)
    return sorted(images, key=sort_key)

with open(output_file, "w", encoding="utf-8") as f_out:
    for _, row in df.iterrows():
        artifact_id = str(row.get("ID", ""))
        description = row.get("Description", "") if pd.notna(row.get("Description")) else ""

        # Skip if no description
        if not description:
            continue

        # Get available images for this artifact
        available_images = artifact_images.get(artifact_id, [])
        sorted_images = sort_images(available_images)[:10]  # Max 10 images

        # Skip if no images available
        if not sorted_images:
            continue

        # Create image URLs
        hf_images = []
        for img_filename in sorted_images:
            hf_images.append({
                "type": "image_url",
                "image_url": {"url": hf_prefix + img_filename}
            })

        # Build conversation with English prompts
        messages = [
            {
                "role": "system",
                "content": "You are a museum curator who creates precise and academically accurate descriptions of archaeological artifacts. Use a formal, scholarly style appropriate for museum catalogs."
            },
            {
                "role": "user",
                "content": hf_images + [
                    {"type": "text", "text": "Describe this artifact in detail including its form, decoration, colors, and preservation state."}
                ]
            },
            {
                "role": "assistant",
                "content": description
            }
        ]

        entry = {"messages": messages}
        f_out.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"\nDone! JSONL with HuggingFace image URLs written to {output_file}")


Done! JSONL with HuggingFace image URLs written to /content/drive/My Drive/master_thesis_project/met_data_for_gpt4o_description_only.jsonl


In [ ]:
# Show the first 5 lines of the JSONL file
with open(output_file, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        print(line.strip())

{"messages": [{"role": "system", "content": "You are a museum curator who creates precise and academically accurate descriptions of archaeological artifacts. Use a formal, scholarly style appropriate for museum catalogs."}, {"role": "user", "content": [{"type": "image_url", "image_url": {"url": "https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/239585_primary.jpg"}}, {"type": "text", "text": "Describe this artifact in detail including its form, decoration, colors, and preservation state."}]}, {"role": "assistant", "content": "Gold pendant in the form of a vase."}]}
{"messages": [{"role": "system", "content": "You are a museum curator who creates precise and academically accurate descriptions of archaeological artifacts. Use a formal, scholarly style appropriate for museum catalogs."}, {"role": "user", "content": [{"type": "image_url", "image_url": {"url": "https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/239586_primary.jpg"}}, {"type": "text

In [ ]:
import json

# Show one pretty-printed sample from your JSONL file
with open(output_file, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i == 0:  # just take the first artefact
            obj = json.loads(line)
            print(json.dumps(obj, indent=2, ensure_ascii=False))
            break

{
  "messages": [
    {
      "role": "system",
      "content": "You are a museum curator who creates precise and academically accurate descriptions of archaeological artifacts. Use a formal, scholarly style appropriate for museum catalogs."
    },
    {
      "role": "user",
      "content": [
        {
          "type": "image_url",
          "image_url": {
            "url": "https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/239585_primary.jpg"
          }
        },
        {
          "type": "text",
          "text": "Describe this artifact in detail including its form, decoration, colors, and preservation state."
        }
      ]
    },
    {
      "role": "assistant",
      "content": "Gold pendant in the form of a vase."
    }
  ]
}


In [ ]:
# Analyze the current description data
import json

# Initialize counters
description_lengths = []
char_counts = []
word_counts = []

with open(output_file, "r", encoding="utf-8") as f:
    for line_num, line in enumerate(f, 1):
        try:
            data = json.loads(line)
            # Get the assistant's content (description)
            assistant_content = data['messages'][2]['content']

            char_count = len(assistant_content)
            word_count = len(assistant_content.split())

            char_counts.append(char_count)
            word_counts.append(word_count)
            description_lengths.append((line_num, char_count, word_count, assistant_content[:100] + "..."))

        except (json.JSONDecodeError, KeyError, IndexError) as e:
            print(f"Error parsing line {line_num}: {e}")

# Print statistics
print("=== DESCRIPTION LENGTH ANALYSIS ===")
print(f"Total examples: {len(char_counts)}")
print(f"Average characters: {sum(char_counts)/len(char_counts):.1f}")
print(f"Average words: {sum(word_counts)/len(word_counts):.1f}")
print(f"Min characters: {min(char_counts)}")
print(f"Max characters: {max(char_counts)}")
print(f"Min words: {min(word_counts)}")
print(f"Max words: {max(word_counts)}")
print()

# Count by quality tiers
short_count = sum(1 for wc in word_counts if wc < 50)
medium_count = sum(1 for wc in word_counts if 50 <= wc < 100)
medium_long_count = sum(1 for wc in word_counts if 100 <= wc < 150)  # This variable exists
long_count = sum(1 for wc in word_counts if wc >= 150)

print("=== QUALITY DISTRIBUTION ===")
print(f"Poor (<50 words): {short_count} examples ({short_count/len(word_counts)*100:.1f}%)")
print(f"Medium (50-99 words): {medium_count} examples ({medium_count/len(word_counts)*100:.1f}%)")
print(f"Medium-long (100-149 words): {medium_long_count} examples ({medium_long_count/len(word_counts)*100:.1f}%)")  # FIXED
print(f"Good (150+ words): {long_count} examples ({long_count/len(word_counts)*100:.1f}%)")
print()

# Show extremes
print("=== SHORTEST DESCRIPTIONS (Potential problems) ===")
shortest = sorted(description_lengths, key=lambda x: x[2])[:10]
for line_num, chars, words, preview in shortest:
    print(f"Line {line_num}: {words} words, {chars} chars - {preview}")

print("\n=== LONGEST DESCRIPTIONS (Good examples) ===")
longest = sorted(description_lengths, key=lambda x: x[2], reverse=True)[:10]
for line_num, chars, words, preview in longest:
    print(f"Line {line_num}: {words} words, {chars} chars - {preview}")

=== DESCRIPTION LENGTH ANALYSIS ===
Total examples: 4997
Average characters: 244.9
Average words: 40.0
Min characters: 4
Max characters: 8722
Min words: 1
Max words: 1449

=== QUALITY DISTRIBUTION ===
Poor (<50 words): 3517 examples (70.4%)
Medium (50-99 words): 1036 examples (20.7%)
Medium-long (100-149 words): 298 examples (6.0%)
Good (150+ words): 146 examples (2.9%)

=== SHORTEST DESCRIPTIONS (Potential problems) ===
Line 1207: 1 words, 12 chars - Vase-shaped....
Line 1208: 1 words, 12 chars - Vase-shaped....
Line 1209: 1 words, 12 chars - Vase-shaped....
Line 1210: 1 words, 12 chars - Vase-shaped....
Line 1280: 1 words, 6 chars - Crane....
Line 1507: 1 words, 5 chars - Stag....
Line 1513: 1 words, 7 chars - centaur...
Line 1706: 1 words, 7 chars - Hollow....
Line 1972: 1 words, 7 chars - Draped....
Line 2005: 1 words, 7 chars - Fibula....

=== LONGEST DESCRIPTIONS (Good examples) ===
Line 2191: 1449 words, 8722 chars - Scenes from the life of the Greek hero AchillesThe Acquisition

Large images would be more tokens and cost more. According to https://platform.openai.com/docs/guides/vision-fine-tuning#size :
If you set the detail parameter for an image to low, the image is resized to 512 by 512 pixels and is only represented by 85 tokens regardless of its size. This will reduce the cost of training.

In [ ]:
import os
import json

jsonl_file = "/content/drive/My Drive/master_thesis_project/met_data_for_gpt4o_description_only.jsonl"

# Check file size
file_size_mb = os.path.getsize(jsonl_file) / (1024 * 1024)
print(f"File size: {file_size_mb:.2f} MB")

# Check number of images per example
max_images_allowed = 10
supported_formats = (".jpg", ".jpeg", ".png", ".webp")
max_10_images = True

with open(jsonl_file, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        try:
            example = json.loads(line)
            user_content = example["messages"][1]["content"]
            if not isinstance(user_content, list):
                print(f"Example {i}: user content is not a list")
                continue

            # Extract image URLs
            image_urls = [x["image_url"]["url"] for x in user_content if x.get("type") == "image_url"]

            if len(image_urls) > max_images_allowed:
                print(f"Example {i}: has {len(image_urls)} images (consider reducing to {max_images_allowed})")
                max_10_images = False

            # check formats
            for url in image_urls:
                if not url.lower().endswith(supported_formats):
                    print(f"Example {i}: unsupported image format -> {url}")

        except Exception as e:
            print(f"Error processing example {i}: {e}")

if max_10_images:
    print(f"All the artefacts have maximal 10 images.")

File size: 4.30 MB
All the artefacts have maximal 10 images.


https://community.openai.com/t/gpt-4-vision-preview-fidelity-detail-parameter/477563

add the parameter detail=low into data  

e.g. taken from https://platform.openai.com/docs/guides/vision-fine-tuning#size  

{
  "type": "image_url",
  "image_url": {
    "url": "https://upload.wikimedia.org/wikipedia/commons/3/36/Danbo_Cheese.jpg",
    "detail": "low"
  }
}

In [ ]:
# add detail=low
import json

# Input and output paths
input_file = "/content/drive/My Drive/master_thesis_project/met_data_for_gpt4o_description_only.jsonl"
output_file = "/content/drive/My Drive/master_thesis_project/met_data_for_gpt4o_description_only_detail_low.jsonl"

with open(input_file, "r", encoding="utf-8") as infile, open(output_file, "w", encoding="utf-8") as outfile:
    for line in infile:
        data = json.loads(line)

        # Go through messages
        for message in data.get("messages", []):
            if isinstance(message.get("content"), list):
                for item in message["content"]:
                    if (isinstance(item, dict) and
                        item.get("type") == "image_url" and
                        "image_url" in item):
                        item["image_url"]["detail"] = "low"

        # Write updated data
        outfile.write(json.dumps(data, ensure_ascii=False) + "\n")

print("All image entries updated with detail='low' and saved to:")
print(output_file)

All image entries updated with detail='low' and saved to:
/content/drive/My Drive/master_thesis_project/met_data_for_gpt4o_description_only_detail_low.jsonl


In [2]:
input_file = "/content/drive/My Drive/master_thesis_project/met_data_for_gpt4o_description_only_detail_low.jsonl"
output_file = "/content/drive/My Drive/master_thesis_project/met_data_for_gpt4o_description_only_detail_low_495.jsonl"

count = 495

with open(input_file, "r") as fin, open(output_file, "w") as fout:
    for i, line in enumerate(fin):
        if i >= count:
            break
        fout.write(line)

print("Saved subset:", output_file)

Saved subset: /content/drive/My Drive/master_thesis_project/met_data_for_gpt4o_description_only_detail_low_495.jsonl


In [3]:
output_file = "/content/drive/My Drive/master_thesis_project/met_data_for_gpt4o_description_only_detail_low_495.jsonl"

with open(output_file, "r") as f:
    count = sum(1 for _ in f)

print("Number of entries:", count)

Number of entries: 495


In [4]:
import json

# Show one pretty-printed sample from your JSONL file
with open(output_file, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i == 10:  # just take the first artefact
            obj = json.loads(line)
            print(json.dumps(obj, indent=2, ensure_ascii=False))
            break

{
  "messages": [
    {
      "role": "system",
      "content": "You are a museum curator who creates precise and academically accurate descriptions of archaeological artifacts. Use a formal, scholarly style appropriate for museum catalogs."
    },
    {
      "role": "user",
      "content": [
        {
          "type": "image_url",
          "image_url": {
            "url": "https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/239905_primary.jpg",
            "detail": "low"
          }
        },
        {
          "type": "text",
          "text": "Describe this artifact in detail including its form, decoration, colors, and preservation state."
        }
      ]
    },
    {
      "role": "assistant",
      "content": "Translucent green, appearing black; one handle and base-knob in yellow brown, the other handle in yellow green; trail in opaque white.Inward-sloping rim-disk, with tooling indent underneath; tall, slightly concave, cylindrical neck; sloping sho

### Split the data into train, val and test set in the ratio of 8:1:1

In [5]:
import json
import random
from pathlib import Path

random.seed(42)  # For reproducible splits

input_file = "/content/drive/My Drive/master_thesis_project/met_data_for_gpt4o_description_only_detail_low_495.jsonl"
output_dir = "/content/drive/My Drive/master_thesis_project/gpt4o/met_split_data/description_only_495"
Path(output_dir).mkdir(parents=True, exist_ok=True)

# Load all examples
with open(input_file, "r", encoding="utf-8") as f:
    lines = f.readlines()

total = len(lines)
print(f"Total examples: {total}")

# Shuffle the data
random.shuffle(lines)

# Compute split indices
train_end_idx = int(total * 0.8)
val_end_idx = int(total * 0.9)

train = lines[:train_end_idx]
val = lines[train_end_idx:val_end_idx]
test = lines[val_end_idx:]

# Save splits
splits = {"train": train, "validation": val, "test": test}

for split_name, split_data in splits.items():
    output_file = Path(output_dir) / f"{split_name}.jsonl"
    with open(output_file, "w", encoding="utf-8") as f_out:
        for line in split_data:
            f_out.write(line)
    print(f"{split_name}: {len(split_data)} examples saved to {output_file}")

# Count total images for each split file
print("\n=== Image Counts ===")
for split_name in ["train", "validation", "test"]:
    split_file = Path(output_dir) / f"{split_name}.jsonl"
    total_images = 0

    with open(split_file, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            for message in data.get("messages", []):
                content = message.get("content")
                if isinstance(content, list):
                    for item in content:
                        if isinstance(item, dict) and item.get("type") == "image_url":
                            total_images += 1

    print(f"{split_name}: {total_images} total images")

Total examples: 495
train: 396 examples saved to /content/drive/My Drive/master_thesis_project/gpt4o/met_split_data/description_only_495/train.jsonl
validation: 49 examples saved to /content/drive/My Drive/master_thesis_project/gpt4o/met_split_data/description_only_495/validation.jsonl
test: 50 examples saved to /content/drive/My Drive/master_thesis_project/gpt4o/met_split_data/description_only_495/test.jsonl

=== Image Counts ===
train: 559 total images
validation: 74 total images
test: 72 total images


In [7]:
# make sure the encoding is utf-8 for gpt4o fine-tuning
import chardet

with open("/content/drive/My Drive/master_thesis_project/met_data_for_gpt4o_description_only_detail_low_495.jsonl", "rb") as f:
    raw = f.read(4096)
    print(chardet.detect(raw))

{'encoding': 'ascii', 'confidence': 1.0, 'language': ''}


### Fine-tuning

In [ ]:
!pip install --upgrade openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.4/948.4 kB 15.6 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.107.0
    Uninstalling openai-1.107.0:
      Successfully uninstalled openai-1.107.0


#### load the inspect the data

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
met_data_dir = "/content/drive/My Drive/master_thesis_project/gpt4o/met_split_data/description_only_495"

In [9]:
!wc -l "{met_data_dir}/train.jsonl"
!wc -l "{met_data_dir}/validation.jsonl"
!wc -l "{met_data_dir}/test.jsonl"

396 /content/drive/My Drive/master_thesis_project/gpt4o/met_split_data/description_only_495/train.jsonl
49 /content/drive/My Drive/master_thesis_project/gpt4o/met_split_data/description_only_495/validation.jsonl
50 /content/drive/My Drive/master_thesis_project/gpt4o/met_split_data/description_only_495/test.jsonl


In [10]:
!head -n 10 "{met_data_dir}/train.jsonl"

{"messages": [{"role": "system", "content": "You are a museum curator who creates precise and academically accurate descriptions of archaeological artifacts. Use a formal, scholarly style appropriate for museum catalogs."}, {"role": "user", "content": [{"type": "image_url", "image_url": {"url": "https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/240097_primary.jpg", "detail": "low"}}, {"type": "image_url", "image_url": {"url": "https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/240097_additional_0.jpg", "detail": "low"}}, {"type": "image_url", "image_url": {"url": "https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/240097_additional_1.jpg", "detail": "low"}}, {"type": "text", "text": "Describe this artifact in detail including its form, decoration, colors, and preservation state."}]}, {"role": "assistant", "content": "In vase-painting of mainland Greece, floral and foliate ornament is always contained and precise. This jug 

In [11]:
import json

# Show one pretty-printed sample from your JSONL file
with open(f"{met_data_dir}/train.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i == 5:  # just take the first artefact
            obj = json.loads(line)
            print(json.dumps(obj, indent=2, ensure_ascii=False))
            break

{
  "messages": [
    {
      "role": "system",
      "content": "You are a museum curator who creates precise and academically accurate descriptions of archaeological artifacts. Use a formal, scholarly style appropriate for museum catalogs."
    },
    {
      "role": "user",
      "content": [
        {
          "type": "image_url",
          "image_url": {
            "url": "https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/240194_primary.jpg",
            "detail": "low"
          }
        },
        {
          "type": "text",
          "text": "Describe this artifact in detail including its form, decoration, colors, and preservation state."
        }
      ]
    },
    {
      "role": "assistant",
      "content": "Horizontal and vertical circles; strainer in mouth.Cypriot strainer vases probably were influenced by Phoenician  examples.  They may have been used to strain herb-infused liquids, thus leaving the herbs in the jug."
    }
  ]
}


#### run gpt4o fine-tuning

use the fine-tunable vision model gpt-4o-2024-08-06  
https://platform.openai.com/docs/guides/vision-fine-tuning . But in this link, it says that "Each example can have at most 10 images."

Accoding to https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/fine-tuning-vision :
Vision fine-tuning is supported for gpt-4o version 2024-08-06 and gpt-4.1 version 2025-04-14 models only. In this link, it says "Each example can have at most 64 images." This is written in 2025. but in this link https://platform.openai.com/docs/guides/supervised-fine-tuning gpt-4.1 version 2025-04-14 models are not listed in the vision fine-tuning page.

In [1]:
# initiate openai client

from openai import OpenAI

client = OpenAI(api_key="sk-xxxxxx")

upload the train and validation files

In [13]:
training_file_upload_response = client.files.create(
    file=open(f"{met_data_dir}/train.jsonl", "rb"),
    purpose="fine-tune"
)

validation_file_upload_response = client.files.create(
    file=open(f"{met_data_dir}/validation.jsonl", "rb"),
    purpose="fine-tune"
)

print("training_file_upload_response:", training_file_upload_response)
print("validation_file_upload_response:", validation_file_upload_response)

training_file_upload_response: FileObject(id='file-9JoMujYicnNFBNDJ2ZoiLU', bytes=305175, created_at=1762783238, filename='train.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)
validation_file_upload_response: FileObject(id='file-Lcb8cW2RovLzRY6jZ9XvV5', bytes=38046, created_at=1762783239, filename='validation.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)


create a fine-tuned mode

In [14]:
from datetime import datetime
import re

def generate_finetune_suffix(project_name: str, dataset_name: str, version: str = None) -> str:
    """
    Generates a kebab-case suffix for fine-tuning jobs, where spaces are replaced with hyphens and all letters are lower-case.

    Args:
        project_name (str): Short name for the project, e.g., 'artefacts'.
        dataset_name (str): Name of the dataset, e.g., 'uzh'.
        version (str, optional): Version or date, e.g., 'v1' or '2025-10-15'. Defaults to today.

    Returns:
        str: kebab-case suffix, e.g., 'artefacts-uzh-2025-10-15'
    """
    def to_kebab(text: str) -> str:
        return re.sub(r'\s+', '-', text.strip().lower())

    if version is None:
        version = datetime.today().strftime("%Y-%m-%d")

    parts = [project_name, dataset_name, version]

    return '-'.join(to_kebab(part) for part in parts)


fine_tuning_response = client.fine_tuning.jobs.create(
    training_file=training_file_upload_response.id,
    validation_file=validation_file_upload_response.id,
    suffix=generate_finetune_suffix('artefacts', 'met', 'v2'),
    model="gpt-4o-2024-08-06"

)

fine_tuning_response

FineTuningJob(id='ftjob-KXFSaiXNr0iDTDnhVQYJl3mK', created_at=1762783315, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'), model='gpt-4o-2024-08-06', object='fine_tuning.job', organization_id='org-QVEfkKafjj7yW8YeqB6EfSev', result_files=[], seed=1871923150, status='validating_files', trained_tokens=None, training_file='file-9JoMujYicnNFBNDJ2ZoiLU', validation_file='file-Lcb8cW2RovLzRY6jZ9XvV5', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'))), user_provided_suffix='artefacts-met-v2', usage_metrics=None, shared_with_openai=False, eval_id=None)

check training job status

In [15]:
status_response = client.fine_tuning.jobs.retrieve(fine_tuning_response.id)

status_response

FineTuningJob(id='ftjob-KXFSaiXNr0iDTDnhVQYJl3mK', created_at=1762783315, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier=2.0, n_epochs=3), model='gpt-4o-2024-08-06', object='fine_tuning.job', organization_id='org-QVEfkKafjj7yW8YeqB6EfSev', result_files=[], seed=1871923150, status='running', trained_tokens=None, training_file='file-9JoMujYicnNFBNDJ2ZoiLU', validation_file='file-Lcb8cW2RovLzRY6jZ9XvV5', estimated_finish=1762787634, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier=2.0, n_epochs=3))), user_provided_suffix='artefacts-met-v2', usage_metrics=None, shared_with_openai=False, eval_id=None)

In [2]:
from openai import OpenAI

client = OpenAI(api_key="sk-xxxxx")
status = client.fine_tuning.jobs.retrieve('ftjob-KXFSaiXNr0iDTDnhVQYJl3mK')
status.status

'succeeded'

run inference using fine-tuned model

In [1]:
from openai import OpenAI

met_data_dir = "/content/drive/My Drive/master_thesis_project/gpt4o/met_split_data/description_only_495"

client = OpenAI(api_key="sk-xxxxxx")
status_response = client.fine_tuning.jobs.retrieve('ftjob-KXFSaiXNr0iDTDnhVQYJl3mK')
fine_tuned_model = status_response.fine_tuned_model

In [2]:
fine_tuned_model

'ft:gpt-4o-2024-08-06:university-of-zurich-department-of-history:artefacts-met-v2:CaNx0s5C'

In [6]:
import json
from openai import OpenAI
import re

# Load JSONL file
def load_jsonl(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

test_data = load_jsonl(f"{met_data_dir}/test.jsonl")

completion = client.chat.completions.create(
    model=status_response.fine_tuned_model,
    messages=test_data[0]['messages'][:-1]
)

completion.choices[0].message

ChatCompletionMessage(content='The olpe has two bands of unequal width around the body and a spout in the shape of an animal’s head, above which there is a nude male figure.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)

In [ ]:
completion = client.chat.completions.create(
    model=status_response.fine_tuned_model,
    messages=test_data[1]['messages'][:-1]
)

completion.choices[0].message

ChatCompletionMessage(content='Niedriger ein wenig unregelmässiger Napf mit horizontal gelochten Henkeln. Die breite untere Zone endet gegen die Basis in zwei schmalen Zonen, die jeweils durch einen umlaufenden Reifen abgetrennt sind. Danach folgt unmittelbar der Standring. Ausserhalb der Zonen sind die Henkel mit Streifen verziert. Auf Wandung und Schulter alternierende breitere und schmalere vertikale Barbotinewellen, flankiert von je einem dünnen Weinrankenstiel. Auf dem Fuss eine im Pressverfahren hergestellte Meduse.\n\nFeiner, schlammfarbener, wenig hart gebrannter Ton mit gleichmässig dünnem, glänzend braunschwarzem bis braunrotem Firnis / Glanzton. Barbotine in Form von wenig dünnflüssiger Auflagemasse.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)

In [7]:
import json
from openai import OpenAI
from tqdm import tqdm

# Load JSONL file
def load_jsonl(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def generate_description_only(data, model, max_samples=None):
    predictions = []

    total_samples = len(data) if max_samples is None else min(max_samples, len(data))

    # Simple progress bar
    for i, sample in enumerate(tqdm(data, total=total_samples, desc="Generating descriptions")):
        if max_samples and i >= max_samples:
            break

        messages = sample["messages"]
        ground_description = messages[2]["content"]

        # First image URL
        first_image_url = None
        for c in messages[1]["content"]:
            if c.get("type") == "image_url":
                first_image_url = c["image_url"]["url"]
                break

        # Single turn: Predict description only
        conversation = [messages[0], messages[1]]

        completion = client.chat.completions.create(
            model=model,
            messages=conversation
        )
        prediction_description = completion.choices[0].message.content.strip()

        # Save predictions
        predictions.append({
            "sample_id": i + 1,
            "first_image_url": first_image_url,
            "ground_description": ground_description,
            "prediction_description": prediction_description
        })

    return predictions

In [8]:
# Load test set
test_data = load_jsonl(f"{met_data_dir}/test.jsonl")

# Generate predictions
test_preds = generate_description_only(test_data, fine_tuned_model)

# Print nicely
def print_predictions(preds):
    for r in preds:
        print(json.dumps(r, indent=2, ensure_ascii=False))

print("\n=== Test Set Description Predictions ===")
print_predictions(test_preds)

Generating descriptions: 100%|██████████| 50/50 [03:37<00:00,  4.36s/it]


=== Test Set Description Predictions ===
{
  "sample_id": 1,
  "first_image_url": "https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/240150_primary.jpg",
  "ground_description": "In the Cypro-Archaic II period (600-480 B.C.) Cypriot potters began to create large jugs with, on the shoulder, female figurines holding miniature jugs. The miniature jug served as a spout. The type of vase developed in the region on Marion, where many imported Greek vases have been found. The new type of pottery may have been created in order to compete with the imports.",
  "prediction_description": "The jug is decorated with horizontal bands in black. The figure at the top of the spout is wheel-made."
}
{
  "sample_id": 2,
  "first_image_url": "https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/241065_primary.jpg",
  "ground_description": "The head, which is related to the Kition goddess type, is mold-made and solid; the veil was added by hand. The back is handmad

In [9]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00


In [10]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=bd04ea92eb10d651a66f02bf1abb946dbe1ed446a9fb8f2c73e619284f0a4ace
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [11]:
!pip install bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.4 MB/s eta 0:00:00


In [12]:
import evaluate
import re

# Load metrics
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")

def compute_description_metrics(predictions):
    refs = [p['ground_description'] for p in predictions]
    hyps = [p['prediction_description'] for p in predictions]

    print(f"Evaluating {len(predictions)} samples...")

    # BLEU - FIXED: Use string inputs, not tokenized
    try:
        bleu_score = bleu.compute(predictions=hyps, references=[[ref] for ref in refs])
        print(f"BLEU: {bleu_score['bleu']:.4f}")
    except Exception as e:
        print(f"BLEU failed: {e}")

    # ROUGE (working)
    try:
        rouge_score = rouge.compute(predictions=hyps, references=refs)
        print("ROUGE:", {k: round(v, 4) for k, v in rouge_score.items()})
    except Exception as e:
        print(f"ROUGE failed: {e}")

    # METEOR (working)
    try:
        meteor_score = meteor.compute(predictions=hyps, references=refs)
        print(f"METEOR: {meteor_score['meteor']:.4f}")
    except Exception as e:
        print(f"METEOR failed: {e}")

    try:
        bertscore_score = bertscore.compute(
            predictions=hyps,
            references=refs,
            lang="de"
        )
        avg_f1 = sum(bertscore_score['f1']) / len(bertscore_score['f1'])
        print(f"BERTScore F1: {avg_f1:.4f}")
    except Exception as e:
        print(f"BERTScore failed: {e}")

# Run metrics
compute_description_metrics(test_preds)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


Evaluating 50 samples...
BLEU: 0.0210
ROUGE: {'rouge1': np.float64(0.3218), 'rouge2': np.float64(0.1102), 'rougeL': np.float64(0.2596), 'rougeLsum': np.float64(0.2618)}
METEOR: 0.2480


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

BERTScore F1: 0.7456


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def compute_tfidf_metrics(predictions):
    """TF-IDF based metrics that work better for creative text"""
    refs = [p['ground_description'] for p in predictions]
    hyps = [p['prediction_description'] for p in predictions]

    # TF-IDF Vectorizer with German-specific settings
    vectorizer = TfidfVectorizer(
        max_features=1000,
        min_df=1,  # Include rare archaeological terms
        max_df=0.8,  # Exclude very common words
        stop_words=None,  # Keep all German words
        ngram_range=(1, 2)  # Include bigrams for technical terms
    )

    # Fit on all text
    all_text = refs + hyps
    vectorizer.fit(all_text)

    # Transform references and predictions
    refs_vec = vectorizer.transform(refs)
    hyps_vec = vectorizer.transform(hyps)

    # Compute cosine similarities
    cosine_sims = []
    for i in range(len(refs)):
        sim = cosine_similarity(refs_vec[i], hyps_vec[i])[0][0]
        cosine_sims.append(sim)

    # Additional metrics
    length_ratios = [len(hyp.split()) / len(ref.split()) for hyp, ref in zip(hyps, refs)]

    print("=== TF-IDF METRICS ===")
    print(f"Average Cosine Similarity: {np.mean(cosine_sims):.4f}")
    print(f"Cosine Similarity Std: {np.std(cosine_sims):.4f}")
    print(f"Average Length Ratio: {np.mean(length_ratios):.4f}")
    print(f"Length Ratio Std: {np.std(length_ratios):.4f}")

    # Quality assessment
    good_similarity = sum(1 for sim in cosine_sims if sim > 0.3) / len(cosine_sims)
    good_length = sum(1 for ratio in length_ratios if 0.5 <= ratio <= 2.0) / len(length_ratios)

    print(f"Examples with good similarity (>0.3): {good_similarity:.1%}")
    print(f"Examples with reasonable length (0.5-2.0x): {good_length:.1%}")

    return {
        'cosine_similarities': cosine_sims,
        'length_ratios': length_ratios,
        'avg_cosine': np.mean(cosine_sims),
        'avg_length_ratio': np.mean(length_ratios)
    }

tfidf_high = compute_tfidf_metrics(test_preds)

=== TF-IDF METRICS ===
Average Cosine Similarity: 0.1879
Cosine Similarity Std: 0.1251
Average Length Ratio: 1.9318
Length Ratio Std: 3.8159
Examples with good similarity (>0.3): 10.0%
Examples with reasonable length (0.5-2.0x): 60.0%
